In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from hierarchical_binary_tree import HierarchicalBinaryTree
import torch.optim as optim
from tqdm import tqdm
import random

In [2]:
# Hyperparameters

model_name = 'bert-base-cased' # teacher model
chunk_size = 64
max_seq_length = 512 # matching teacher model
batch_size = 8
num_epochs = 1 # don't want to overfit
learning_rate = 2e-5
weight_decay = 0.01
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

c:\Program Files\Python311\Lib\site-packages\torch\cuda\__init__.py:129: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 1: invalid argument (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
tokenizer = BertTokenizer.from_pretrained(model_name)

In [16]:
# Dataset Loading - TriviaQA

dataset = load_dataset("trivia_qa", "unfiltered", split="train")  
# was train[:1%] - smaller slice for dev

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/47 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/36 [00:00<?, ?it/s]

In [13]:
# def preprocess(example):
#     context = " ".join(example["search_results"]["description"])
#     input_text = example["question"] + " " + context
#     encoding = tokenizer(
#         input_text,
#         padding="max_length",
#         truncation=True,
#         max_length=max_seq_length,
#         return_tensors="pt"
#     )
#     # Label: first token of answer span (if it exists in input)
#     label_pos = 0  # fallback
#     if example["answer"]["value"] in example["search_results"]["description"]:
#         answer_start = example["search_results"]["description"].index(example["answer"]["value"])
#         before = tokenizer(example["search_results"]["description"][:answer_start], truncation=True, max_length=max_seq_length)["input_ids"]
#         label_pos = min(len(before), max_seq_length - 1)
#     encoding["label_pos"] = torch.tensor(label_pos)
#     return encoding

# dataset = dataset.map(preprocess)
# dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label_pos'])

# train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [6]:
# def preprocess(example):
#     descriptions = example["search_results"]["description"]
#     context = " ".join(descriptions[:3]) if isinstance(descriptions, list) else descriptions
#     input_text = example["question"] + " " + context

#     encoding = tokenizer(
#         input_text,
#         padding="max_length",
#         truncation=True,
#         max_length=max_seq_length,
#     )

#     label_pos = 0
#     answer_text = example["answer"]["value"]

#     if answer_text in context:
#         answer_start = context.index(answer_text)
#         before = context[:answer_start]
#         before_ids = tokenizer(before, truncation=True, max_length=max_seq_length)["input_ids"]
#         label_pos = min(len(before_ids), max_seq_length - 1)

#     encoding["label_pos"] = torch.tensor(label_pos)
#     return encoding


In [17]:
def preprocess(example):
    descriptions = example["search_results"]["description"]
    context = " ".join(descriptions[:3]) if isinstance(descriptions, list) else descriptions
    input_text = example["question"] + " " + context

    encoding = tokenizer(
        input_text,
        padding="max_length",
        truncation=True,
        max_length=max_seq_length,
    )
    
    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    
    # Prepend CLS token
    cls_token_id = tokenizer.cls_token_id
    input_ids = [cls_token_id] + input_ids[:-1]  # replace last token to keep length
    attention_mask = [1] + attention_mask[:-1]
    
    encoding["input_ids"] = torch.tensor(input_ids)
    encoding["attention_mask"] = torch.tensor(attention_mask)

    label_pos = 0
    answer_text = example["answer"]["value"]

    if answer_text in context:
        answer_start = context.index(answer_text)
        before = context[:answer_start]
        before_ids = tokenizer(before, truncation=True, max_length=max_seq_length)["input_ids"]
        label_pos = min(len(before_ids) + 1, max_seq_length - 1)  # +1 for CLS token shift

    encoding["label_pos"] = torch.tensor(label_pos)
    return encoding

In [18]:
preprocessed_dataset = dataset.map(preprocess)
preprocessed_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label_pos'])

train_loader = DataLoader(preprocessed_dataset, batch_size=batch_size, shuffle=True)

Map:   0%|          | 0/87622 [00:00<?, ? examples/s]

In [19]:
# Teacher Model (Frozen BERT)
teacher_model = BertModel.from_pretrained(model_name, output_attentions=True).to(device)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False

In [20]:
# Student Model

model = HierarchicalBinaryTree(
    model_name=model_name,
    chunk_size=chunk_size
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [ ]:
# Training Loop - 1% of the training data

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for i, batch in enumerate(progress):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label_pos = batch['label_pos'].to(device)

        with torch.no_grad():
            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            teacher_attn = torch.stack(teacher_outputs.attentions, dim=1)  # (B, Layers, Heads, T, T)
            teacher_attn = teacher_attn[:, -1, :, 0]  # use last layer, CLS as query
            teacher_attn = teacher_attn.mean(dim=1)  # avg over heads → (B, T)

        loss = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            label_pos=label_pos,
            teacher_attn=teacher_attn.unsqueeze(1)  # (B, 1, T)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        print("Batch Loss:", loss.item())
        progress.set_postfix(loss=total_loss / (progress.n + 1))

    print(f"Epoch {epoch+1} average loss: {total_loss / len(train_loader):.4f}")


Epoch 1:   1%|          | 1/110 [00:21<38:29, 21.19s/it, loss=76.6]

Batch Loss: 76.55315399169922


Epoch 1:   2%|▏         | 2/110 [00:44<40:29, 22.50s/it, loss=80.2]

Batch Loss: 83.89768981933594


Epoch 1:   3%|▎         | 3/110 [01:05<39:02, 21.89s/it, loss=80.1]

...

Batch Loss: 15.67839241027832


Epoch 1:  99%|█████████▉| 109/110 [48:56<00:22, 22.97s/it, loss=21.4]

Batch Loss: 16.48655128479004


Epoch 1: 100%|██████████| 110/110 [49:09<00:00, 26.82s/it, loss=21.3]

Batch Loss: 15.006429672241211
Epoch 1 average loss: 21.3273


In [14]:
len(progress)

110

In [ ]:
checkpoint_path = "checkpoint.pth"

torch.save({
    'batch': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    # add other info if you want, e.g. loss or scheduler state
}, checkpoint_path)

print(f"Checkpoint saved to {checkpoint_path}")


Checkpoint saved to checkpoint.pth


In [ ]:
# Training Loop - now with all the data

training_progress = "batch_losses.csv"

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for i, batch in enumerate(progress):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label_pos = batch['label_pos'].to(device)

        with torch.no_grad():
            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            teacher_attn = torch.stack(teacher_outputs.attentions, dim=1)  # (B, Layers, Heads, T, T)
            teacher_attn = teacher_attn[:, -1, :, 0]  # use last layer, CLS as query
            teacher_attn = teacher_attn.mean(dim=1)  # avg over heads → (B, T)

        loss = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            label_pos=label_pos,
            teacher_attn=teacher_attn.unsqueeze(1)  # (B, 1, T)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # print("Batch Loss:", loss.item())
        with open(training_progress, 'a') as f:
            f.write(f"{loss.item()}\n")
        progress.set_postfix(loss=total_loss / (progress.n + 1))

    print(f"Epoch {epoch+1} average loss: {total_loss / len(train_loader):.4f}")

    checkpoint_path = f"checkpoint_{epoch}_1.pth"
    torch.save({
        'batch': i,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        # add other info if you want, e.g. loss or scheduler state
    }, checkpoint_path)

    print(f"Checkpoint saved to {checkpoint_path}")



Epoch 1:   0%|          | 28/10953 [11:04<66:10:59, 21.81s/it, loss=15.8]

In [ ]:
# Saved when I stopped training after almost 1000 minutes; then I restarted training loop above with same model and hyperparams
checkpoint_path = f"checkpoint_{epoch}.pth"
torch.save({
    'batch': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    # add other info if you want, e.g. loss or scheduler state
}, checkpoint_path)

print(f"Checkpoint saved to {checkpoint_path}")

Checkpoint saved to checkpoint_0.pth
